In [4]:
import os
import json
import re
from unstructured.partition.pdf import partition_pdf

In [5]:
def infer_subsection_from_section(section_value):
    text = str(section_value or "").strip()
    if ":" not in text:
        return text or "Main Text", None

    left, right = text.split(":", 1)
    left = left.strip()
    right = right.strip()
    if left and right:
        return left, right
    return text or "Main Text", None



def build_hierarchy_metadata(pdf_path, section, subsection=None, extra_metadata=None):
    metadata = {
        "source": os.path.basename(pdf_path),
        "section": section or "Main Text",
    }
    metadata.update(extra_metadata or {})

    level_1 = metadata.get("path_level_1") or metadata.get("title") or metadata["source"]
    level_2 = metadata.get("path_level_2") or metadata.get("section") or "Main Text"
    level_3 = metadata.get("path_level_3") or metadata.get("subsection") or subsection

    inferred_section, inferred_subsection = infer_subsection_from_section(level_2)
    level_2 = inferred_section or "Main Text"
    if not level_3:
        level_3 = inferred_subsection

    # Keep a 3-level schema for every chunk.
    level_3 = str(level_3).strip() if level_3 else "General"

    metadata["section"] = str(level_2)
    metadata["subsection"] = level_3
    metadata["path_level_1"] = str(level_1)
    metadata["path_level_2"] = str(level_2)
    metadata["path_level_3"] = level_3
    metadata["hierarchy_path"] = f"{metadata['path_level_1']} -> {metadata['path_level_2']} -> {metadata['path_level_3']}"
    metadata["path_depth"] = 3
    return metadata



def parse_pdf_hierarchically(pdf_path, extra_metadata=None):
    """
    Reads a PDF, detects its visual layout, and groups paragraphs
    under their respective section headers.
    """
    if extra_metadata is None:
        extra_metadata = {}

    print(f" Processing: {os.path.basename(pdf_path)}... (This might take a minute on the first run)")

    try:
        elements = partition_pdf(
            filename=pdf_path,
            strategy="hi_res",
            infer_bounding_boxes=True
        )
    except Exception as e:
        print(f"Error parsing {pdf_path}: {e}")
        return []

    document_chunks = []
    current_section = "Abstract / Introduction"
    current_subsection = None

    for element in elements:
        element_type = element.category
        clean_text = str(element.text).replace("\n", " ").strip()

        # Update section tracker when a new header/title is encountered.
        if element_type in ["Title", "Header"]:
            if len(clean_text) > 3:
                depth = getattr(getattr(element, "metadata", None), "category_depth", None)
                if isinstance(depth, int) and depth >= 1:
                    current_subsection = clean_text
                elif ":" in clean_text and current_section:
                    _, inferred_sub = infer_subsection_from_section(clean_text)
                    current_subsection = inferred_sub or clean_text
                else:
                    current_section = clean_text
                    current_subsection = None

        elif element_type == "Table":
            table_html = getattr(element.metadata, "text_as_html", None) or clean_text
            metadata = build_hierarchy_metadata(pdf_path, current_section, current_subsection, extra_metadata)
            document_chunks.append({
                "type": "table",
                "text": table_html,
                "metadata": metadata,
            })

        elif element_type in ["NarrativeText", "Text", "ListItem"]:
            if len(clean_text) > 40:
                metadata = build_hierarchy_metadata(pdf_path, current_section, current_subsection, extra_metadata)
                document_chunks.append({
                    "type": "text",
                    "text": clean_text,
                    "metadata": metadata,
                })

    return document_chunks

In [6]:
# --- Test the Pipeline ---
papers_dir = "./dataset"
metadata_file = "metadata.json"

# Make sure the directory exists
if not os.path.exists(papers_dir):
    print(f"Could not find the '{papers_dir}' directory.")
    exit()

# Load the external metadata mapping
with open(metadata_file, "r") as f:
    pdf_metadata_mapping = json.load(f)

# Get the PDFs in the folder
pdf_files = [f for f in os.listdir(papers_dir) if f.endswith('.pdf')]

if not pdf_files:
    print(" No PDFs found in the 'papers' folder!")
    exit()

all_chunks = []
for pdf_file in pdf_files:
    pdf_path = os.path.join(papers_dir, pdf_file)
    extra_meta = pdf_metadata_mapping.get(pdf_file, {})
    chunks = parse_pdf_hierarchically(pdf_path, extra_metadata=extra_meta)
    all_chunks.extend(chunks)

print(f"\n Successfully extracted {len(all_chunks)} contextual chunks across all documents!")
print("-" * 50)
print("Here is a sample of what your parsed data looks like:\n")

# Print a sample of the first 3 chunks to verify
for i, chunk in enumerate(all_chunks[5:8]): # Skipping first few in case of title pages
    print(json.dumps(chunk, indent=2))

# Save the parsed chunks to a JSON file for embedding later
with open("chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)
print("\nSaved chunks to chunks.json")

 Processing: p3.pdf... (This might take a minute on the first run)
 Processing: p1.pdf... (This might take a minute on the first run)
 Processing: p2.pdf... (This might take a minute on the first run)

 Successfully extracted 526 contextual chunks across all documents!
--------------------------------------------------
Here is a sample of what your parsed data looks like:

{
  "type": "text",
  "text": "\u2022 SVAMP: math word problems with varying structures (Patel et al., 2021)",
  "metadata": {
    "source": "p3.pdf",
    "section": "Arithmetic Reasoning",
    "title": "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
    "author": "Charlie Academic",
    "year": 2022,
    "topic": "Computer Vision",
    "subsection": "General",
    "path_level_1": "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
    "path_level_2": "Arithmetic Reasoning",
    "path_level_3": "General",
    "hierarchy_path": "Chain-of-Thought Prompting Elicits Reasoning